# Conditional richness: Section 5
Read the prepared cluster table. Data-only mode is the default: no fits are needed or run. This notebook never recomputes richness.
The data-only offset is $d=\log_{10}(\lambda_{\rm RM}/\lambda_{\rm spec})$ in dex, relative to equality, not a fitted residual or a pure projection measurement. Offset error bars show the SEM of $d$; scatter error bars show the SEM in log richness transformed to linear units.

The models use the manuscript symbols $A,B,C,\sigma_0$ and $f_{\rm proj},\tau,\sigma_\lambda$. The redshift bins are $[0.10,0.18)$, $[0.18,0.24)$, and $[0.24,0.35)$.
The lognormal $\mu_{\rm RM}$ is the unselected median; the mixture $\mu_{\rm RM}$ is the unboosted Gaussian core. Dispersions are conditional on measured spectroscopic richness, not deconvolved intrinsic scatter.

Run `prepare_conditional_richness.py` first. Only set `DATA_ONLY = False` after generating saved fits with `fit_conditional_richness_mpi.py`. See `CONDITIONAL_RICHNESS.md` for commands and assumptions.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

REPO_ROOT = Path('/global/homes/z/zzhang13/DESI/Projection')
if not REPO_ROOT.exists():
    REPO_ROOT = Path.cwd().resolve()
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'richness_relation').is_dir():
        REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'richness_relation').is_dir():
    raise FileNotFoundError('Set REPO_ROOT to the DESI/Projection checkout')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from richness_relation.plot_conditional_richness import plot_all
DATA_PATH = REPO_ROOT / 'catalogs/conditional_richness/sample.npz'
FIT_ROOT = REPO_ROOT / 'catalogs/conditional_richness/fits'
OUTPUT_DIR = REPO_ROOT / 'plots/conditional_richness'
DATA_ONLY = True
SEED = 42
BAND_DRAWS = 30
audit = json.loads(DATA_PATH.with_suffix('.json').read_text())
display(audit['cutflow'])
for warning in audit['warnings']:
    print(warning)

In [ ]:
comparison = plot_all(DATA_PATH, FIT_ROOT, OUTPUT_DIR, SEED, BAND_DRAWS, data_only=DATA_ONLY)
FIGURE_DIR = OUTPUT_DIR / 'data_only' if DATA_ONLY else OUTPUT_DIR
if not DATA_ONLY:
    display(comparison)
parameter_file = OUTPUT_DIR / 'parameter_summary.csv'
if not DATA_ONLY and parameter_file.exists():
    display(pd.read_csv(parameter_file))

In [ ]:
names = (['richness_relation_redshift_bins', 'richness_logratio_offsets'] if DATA_ONLY else
         ['richness_relation_redshift_bins', 'conditional_richness_distributions',
          'lognormal_conditional_residuals', 'mixture_conditional_residuals'])
for name in names:
    path = FIGURE_DIR / f'{name}.png'
    if path.exists():
        display(Image(filename=str(path)))

## Fit diagnostics
AIC/BIC differences are computed only within common samples. They are supporting comparisons, not Bayesian evidence. PPC probabilities compare replicated and observed discrepancies at each posterior draw; they are not classical chi-squared p-values.
Check residual trends, tail fractions, individual chains, independent-ensemble agreement, autocorrelation times, and prior boundaries before reporting parameters. A short smoke test is not a scientific posterior.

In [ ]:
fit_paths = [] if DATA_ONLY else sorted(FIT_ROOT.glob(f'*/seed_{SEED}/summary.json'))
for path in fit_paths:
    result = json.loads(path.read_text())
    print(result['task'])
    display(pd.DataFrame(result.get('chain_diagnostics', [])))
    display(pd.DataFrame(result.get('posterior_predictive', {})).T)
    for suffix in ['traces', 'posterior_marginals']:
        figure = OUTPUT_DIR / f"{result['task']}_{suffix}.png"
        if figure.exists():
            display(Image(filename=str(figure)))